# 🧠 Bilgisayarlı Görüye Giriş (Computer Vision)

Hoş geldiniz! Bu notebook'ta, makinelerin dünyayı nasıl gördüğünü anlamak için adım adım uygulamalar yapacağız.
Bilgisayarların dünyasında resimler yoktur; sadece sayılar ve matrisler vardır. Bu notebook'ta hem teknik detayları 
öğrenecek hem de bu işlemlerin arkasındaki temel mantığı kavrayacağız.

> 💡 **Not:** Başlamadan önce `data/` klasöründe görsellerinizin olduğundan emin olun.

## Gereksinimleri indir

In [ ]:
!pip install opencv-python mediapipe numpy matplotlib

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import matplotlib.pyplot as plt
import numpy as np
import urllib.request
import os

# Matplotlib ayarları - Grafikleri daha büyük göstermek için
plt.rcParams['figure.figsize'] = [10, 6]

## 🧐 Bölüm 1: Görüntülerin Doğası (Makineler Ne Görür?)

İnsanlar bir ekrana baktığında şekiller, renkler, yüzler görür. Oysa makine sadece devasa bir sayı matrisi görür.
Her bir piksel, 0 ile 255 arasında bir değer alır. (0 = Siyah, 255 = Beyaz).
Renkli görüntülerde bu pikseller 3 farklı 'kanala' ayrılır: **Kırmızı (R), Yeşil (G) ve Mavi (B)**.

Şimdi bir resmi yükleyip bu matrisi inceleyelim.

In [ ]:
# OpenCV resmi BGR (Blue, Green, Red) formatında okur.
resim_bgr = cv2.imread('../data/insan_yuzu.png') # Doğru resmi okuduğunuzdan emin olun

if resim_bgr is not None:
    # Biz insanların alışık olduğu RGB (Red, Green, Blue) formatına çevirelim
    resim_rgb = cv2.cvtColor(resim_bgr, cv2.COLOR_BGR2RGB)
    
    # Resmin boyutlarını (shape) görelim:
    print(f"Resmin Boyutu (Yükseklik, Genişlik, Kanal): {resim_rgb.shape}")
    print(f"Toplam Piksel Sayısı: {resim_rgb.shape[0] * resim_rgb.shape[1]}")
    
    # Matrisin çok ufak bir kısmını (örneğin ilk 3x3 piksellik alanını) yazdıralım:
    print("\nİlk 3x3 pikselin RGB değerleri:\n", resim_rgb[0:3, 0:3])
    
    plt.imshow(resim_rgb)
    plt.title('Orijinal Resim (RGB)')
    plt.axis('off')
    plt.show()
else:
    print("HATA: Resim bulunamadı. Lütfen '../data/insan_yuzu.png' yolunu kontrol edin.")

## 🖌️ Bölüm 2: Temel Görüntü İşleme (Filtreler ve Kenar Tespiti)

Görüntü işleme, pikselleri özel algoritmalarla (kernel/filtre) değiştirerek onlardan anlamlı bazı taslaklar çıkarma işidir.
Bunun en güzel örneklerinden biri **Kenar Tespiti**'dir.

Kenar bulmak, makine için nesnelerin nerede başlayıp bittiğini anlamanın ilk adımıdır.

In [ ]:
if resim_bgr is not None:
    # 1. Adım: Resmi gri tonlamaya çevirmek (Hesaplanacak veri miktarını 3 kanaldan 1 kanala düşürür)
    gri_resim = cv2.cvtColor(resim_bgr, cv2.COLOR_BGR2GRAY)
    
    # 2. Adım: Gürültüyü azaltmak için bulanıklaştırma (Gaussian Blur)
    # Bu sayede sadece belirgin kenarları tespit etmiş oluruz.
    bulanik_resim = cv2.GaussianBlur(gri_resim, (5, 5), 0)
    
    # 3. Adım: Canny Algoritması ile Kenar Tespiti
    # 100 ve 200, alt ve üst eşik değerleridir. Sayılarla oynayarak detayları artırıp azaltabilirsiniz.
    kenarlar = cv2.Canny(bulanik_resim, 100, 200)
    
    # Sonuçları Yan Yana Gösterelim
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    
    ax[0].imshow(gri_resim, cmap='gray')
    ax[0].set_title('Gri Tonlamalı Resim')
    ax[0].axis('off')
    
    ax[1].imshow(kenarlar, cmap='gray')
    ax[1].set_title('Canny Edge Detection (Kenar Tespiti)')
    ax[1].axis('off')
    
    plt.show()

## 🧑‍💻 Bölüm 3: Gelenekselden Derin Öğrenmeye - Yüz Tespiti

Sadece kenarları bulmak yetmez. Bir yapay zeka uygulamasının resimdeki "kavramları" (insan, araba, ağaç) anlaması gerekir.
Eskiden (2000'lerin başı) yüz tespiti piksellerdeki siyah-beyaz oranlarına (Haar Cascade) dayanırdı.
Günümüzde ise **MediaPipe** gibi Google'ın geliştirdiği hafifleştirilmiş derin öğrenme altyapıları ile çok kısıtlı cihazlarda bile anında tespit yapılabiliyor.

Şimdi MediaPipe Face Detection ile yüzleri statik bir görsel üzerinde tespit edelim.

In [ ]:
# Model dosyalarını otomatik indir
def model_indir(url, dosya_adi):
    if not os.path.exists(dosya_adi):
        print(f"{dosya_adi} indiriliyor...")
        urllib.request.urlretrieve(url, dosya_adi)
        print(f"{dosya_adi} indirildi!")

In [ ]:
# Model dosyasını otomatik indir (eğer yoksa)
model_path = 'blaze_face_short_range.tflite'
url = 'https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite'
model_indir(url, model_path)

if resim_bgr is not None:
    # BGR'den RGB'ye çevir
    resim_rgb = cv2.cvtColor(resim_bgr, cv2.COLOR_BGR2RGB)
    cizim_resmi = resim_rgb.copy()
    
    # MediaPipe Face Detector ayarları
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.FaceDetectorOptions(
        base_options=base_options,
        min_detection_confidence=0.5
    )
    
    # Detector oluştur
    detector = vision.FaceDetector.create_from_options(options)
    
    # MediaPipe Image formatına çevir
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=resim_rgb)
    
    # Yüz tespiti yap
    sonuclar = detector.detect(mp_image)
    
    if sonuclar.detections:
        print(f"{len(sonuclar.detections)} adet yüz tespit edildi!")
        
        # Her bir yüz için kutu çiz
        for detection in sonuclar.detections:
            bbox = detection.bounding_box
            
            # Bounding box koordinatları
            x = bbox.origin_x
            y = bbox.origin_y
            w = bbox.width
            h = bbox.height
            
            # Dikdörtgen çiz (yeşil renk)
            cv2.rectangle(cizim_resmi, (x, y), (x + w, y + h), (0, 255, 0), 3)
            
            # Güven skorunu yaz
            skor = detection.categories[0].score
            cv2.putText(cizim_resmi, f'{skor:.2f}', (x, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            
            # Kilit noktalarını (keypoints) çiz
            for keypoint in detection.keypoints:
                kp_x = int(keypoint.x * resim_rgb.shape[1])
                kp_y = int(keypoint.y * resim_rgb.shape[0])
                cv2.circle(cizim_resmi, (kp_x, kp_y), 5, (255, 0, 0), -1)
    else:
        print("Yüz tespit edilemedi.")
    
    # Sonucu göster
    plt.figure(figsize=(10, 8))
    plt.imshow(cizim_resmi)
    plt.title('MediaPipe ile Derin Öğrenme Tabanlı Yüz Tespiti')
    plt.axis('off')
    plt.show()
else:
    print("Resim yüklenemedi! Dosya yolunu kontrol edin.")

## 🌐 Bölüm 4: Modern Görüş (Holistic Modeling / Poz Tespiti)

MediaPipe sadece yüzleri tespit etmekle kalmaz; elleri, iskelet yapısını (pose) ve hatta karmaşık yüz hatlarını aynı anda analiz edebilir. Buna bütüncül (Holistic) analiz denir.
Donanımı zayıf bir cihazda ağar modeller (örn: YOLO veya büyük Transformerlar) çalıştırmak uzun sürebilir ve kaynakları tüketebilir. Fakat optimize edilmiş MediaPipe çözümleri CPU üzerinde bile çok hızlı çalışır.

Aşağıda bir insanın hem yüz hatlarını hem de iskeletini algılayan basit ama etkili modelin statik bir fotoğrafta nasıl çalıştığını göreceğiz.

In [ ]:
# Pose ve Face Landmark modelleri
pose_model = 'pose_landmarker_heavy.task'
face_model = 'face_landmarker.task'

model_indir('https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task', pose_model)
model_indir('https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task', face_model)

# Resmi yükle
resim_bgr_2 = cv2.imread('../data/insan_yuzu.png')  # Kendi resim yolunuzu yazın

if resim_bgr_2 is not None:
    resim_rgb_2 = cv2.cvtColor(resim_bgr_2, cv2.COLOR_BGR2RGB)
    resim_cizimli = resim_rgb_2.copy()
    
    # MediaPipe Image formatına çevir
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=resim_rgb_2)
    
    # ========== POSE (İSKELET) TESPİTİ ==========
    pose_options = vision.PoseLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=pose_model),
        output_segmentation_masks=False
    )
    pose_detector = vision.PoseLandmarker.create_from_options(pose_options)
    pose_results = pose_detector.detect(mp_image)
    
    # Pose bağlantıları (iskelet çizgileri)
    POSE_CONNECTIONS = [
        (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
        (9, 10), (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21),
        (17, 19), (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
        (11, 23), (12, 24), (23, 24), (23, 25), (24, 26), (25, 27), (26, 28),
        (27, 29), (28, 30), (29, 31), (30, 32), (27, 31), (28, 32)
    ]
    
    h, w = resim_rgb_2.shape[:2]
    
    # Pose landmarkları çiz
    if pose_results.pose_landmarks:
        for pose_landmarks in pose_results.pose_landmarks:
            # Bağlantı çizgilerini çiz
            for connection in POSE_CONNECTIONS:
                start_idx, end_idx = connection
                if start_idx < len(pose_landmarks) and end_idx < len(pose_landmarks):
                    start = pose_landmarks[start_idx]
                    end = pose_landmarks[end_idx]
                    start_point = (int(start.x * w), int(start.y * h))
                    end_point = (int(end.x * w), int(end.y * h))
                    cv2.line(resim_cizimli, start_point, end_point, (0, 255, 0), 2)
            
            # Noktaları çiz
            for landmark in pose_landmarks:
                x = int(landmark.x * w)
                y = int(landmark.y * h)
                cv2.circle(resim_cizimli, (x, y), 5, (255, 0, 0), -1)
        print("İskelet tespit edildi!")
    else:
        print("İskelet tespit edilemedi.")
    
    # ========== FACE MESH (YÜZ AĞI) TESPİTİ ==========
    face_options = vision.FaceLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=face_model),
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        num_faces=1
    )
    face_detector = vision.FaceLandmarker.create_from_options(face_options)
    face_results = face_detector.detect(mp_image)
    
    # Yüz landmarkları çiz
    if face_results.face_landmarks:
        for face_landmarks in face_results.face_landmarks:
            for landmark in face_landmarks:
                x = int(landmark.x * w)
                y = int(landmark.y * h)
                cv2.circle(resim_cizimli, (x, y), 1, (0, 255, 255), -1)
        print("Yüz mesh tespit edildi!")
    else:
        print("Yüz mesh tespit edilemedi.")
    
    # Sonucu göster
    plt.figure(figsize=(12, 10))
    plt.imshow(resim_cizimli)
    plt.title('MediaPipe: Yüz Mesh ve İskelet Analizi')
    plt.axis('off')
    plt.show()
else:
    print("Resim yüklenemedi! Dosya yolunu kontrol edin.")